# TOD Visualization

Carrega o **último registro** do repositório de simulações (Postgres + dataset
persistido em HDF5/Zarr) e gera a visualização interativa do Time-Ordered Data
(waterfall + séries temporais). Execute `50_TOD_SIMULATION.ipynb` antes.

In [ ]:
"""Imports e setup do ambiente."""

from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nest_asyncio
import numpy as np  # noqa: F401

# ── Detecta raiz do projeto e adiciona src/ ao path ANTES dos imports do pacote ──
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
project_root = None
for p in candidates:
    if (p / "src" / "radioskysims").exists() and (p / "settings.toml").exists():
        project_root = p
        break

if project_root is None:
    msg = "Raiz do projeto não encontrada."
    raise RuntimeError(msg)

src_path = str((project_root / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# ── Imports do projeto (após configurar sys.path) ──
from radioskysims.application.services.sim_orchestrator import SimOrchestrator

nest_asyncio.apply()

In [ ]:
# ── Carrega o ÚLTIMO registro do repositório de simulações ──
# O repositório (Postgres) persiste o resumo de cada simulação e o caminho do
# dataset gravado (HDF5/Zarr). latest() devolve o resumo mais recente; load(id)
# recompõe o SimulationOrchestratorResult (dataset do arquivo + resumo do banco)
# e o define como "item corrente" (repo.current).
orch = SimOrchestrator.from_settings("bingo")
repo = orch.simulation_repository

latest = repo.latest()
if latest is None:
    msg = (
        "Nenhum registro de simulação no banco. "
        "Execute 50_TOD_SIMULATION.ipynb antes da visualização."
    )
    raise RuntimeError(msg)

sim_id = latest["_id"]
loaded = repo.load(sim_id)
comp_result = loaded  # SimulationOrchestratorResult

if comp_result.simulation_result is None:
    msg = (
        "Dataset persistido não encontrado (filepath ausente/inválido no registro). "
        "Reexecute 50_TOD_SIMULATION.ipynb."
    )
    raise RuntimeError(msg)

ds_orch = comp_result.simulation_result.dataset
tr = comp_result.simulation_result.time_range
fr = comp_result.simulation_result.freq_range

print(f"🔎 Último registro carregado: id={sim_id} | criado em {latest.get('_created_at')}")
print(f"   filepath={latest.get('computation', {}).get('filepath')}")
print(f"ℹ️  dims={dict(ds_orch.sizes)}")
print(f"   tr={tr.start_jd:.4f}→{tr.end_jd:.4f} | freq={fr.min_mhz:.1f}–{fr.max_mhz:.1f} MHz")
print(comp_result.format_summary())

In [ ]:
import holoviews as hv

hv.extension("bokeh")
try:
    from radioskysims.visualization import (
        serve_tod_panel,
        time_series_plot,
        waterfall_plot,
    )

    # ── Painel interativo (seletores no topo) ──
    # Antenna (pointing) + Polarização + Tipos de fonte + switch Tempo
    # (UTC/Local) + switch Mostrar eventos. Observação: os eventos (trânsitos)
    # NÃO são persistidos no repositório — o painel renderiza o waterfall sem
    # o overlay de eventos. O widget Panel falha no render do VS Code; serve
    # via pn.serve → abra a URL no navegador.
    url, server = serve_tod_panel(
        ds_orch,
        events=None,
        port=5007,
        title="Waterfall TOD interativo",
        width=1200,
        height=350,
        cmap="inferno",
        use_datashader=True,
        time_range=tr,
    )

    # Encerrar servidores ao re-executar? Descomente se quiser manter ativo.
    # stop_waterfall_servers()

except ImportError as e:
    print(f"⚠️  {e}")
    print("Instale com: pip install holoviews datashader")
except Exception as e:
    import traceback

    traceback.print_exc()
    print(f"⚠️  Erro ao gerar visualização: {e}")
    print("(HoloViews/Datashader podem não estar instalados)")